# Session 4 — Gradient Descent, Hyperparameters, and Feature Scaling
### Day 1 | FDP: Machine Learning for Materials and Metallurgical Engineering

Session 3 fit Hall-Petch and Cu using `np.polyfit` — a closed-form solver that finds the best (σ0, k) directly. Today we build the **alternative, iterative method** ourselves: gradient descent. We'll use it to understand *why* it needs careful tuning (hyperparameters), and *why* feature scaling matters — using the same small, familiar Hall-Petch dataset throughout so every step is easy to track.

**This session has four parts:**
- **Part A** — Deriving and implementing gradient descent by hand
- **Part B** — Convergence & convexity (why gradient descent is guaranteed to work here)
- **Part C** — Hyperparameters: learning rate, batch size, epochs
- **Part D** — Multiple features & feature scaling

## Setup — Import Libraries and Recall the Hall-Petch Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.array([0.0018, 0.0025, 0.004, 0.007, 0.016, 0.060, 0.25])
sigma_y = np.array([530, 450, 380, 300, 230, 155, 115])

x = d ** -0.5     # the linearized feature, same as Day 1 morning
y = sigma_y
n = len(x)

# Ground truth, from Day 1 morning's np.polyfit
k_true, sigma0_true = np.polyfit(x, y, 1)
print(f"Ground truth (np.polyfit): sigma_0 = {sigma0_true:.2f}, k = {k_true:.2f}")
print(f"Feature range (d^-0.5): {x.min():.2f} to {x.max():.2f}")


---
## Part A — Deriving Gradient Descent by Hand

### The Cost Function

We measure fit quality the same way as Session 3 — mean squared error — but now written as a function of the two parameters we're searching for, $\sigma_0$ and $k$:

$$J(\sigma_0, k) = \frac{1}{n}\sum_{i=1}^{n} \left(y_i - (\sigma_0 + k\,x_i)\right)^2$$

### The Update Rule

Gradient descent repeatedly nudges $\sigma_0$ and $k$ in the direction that reduces $J$ the fastest — found by taking the partial derivative of $J$ with respect to each parameter:

$$\frac{\partial J}{\partial k} = -\frac{2}{n}\sum x_i\,(y_i - \hat{y}_i) \qquad\qquad \frac{\partial J}{\partial \sigma_0} = -\frac{2}{n}\sum (y_i - \hat{y}_i)$$

Then both parameters are updated **simultaneously**, scaled by a small step size (the *learning rate*, $\alpha$):

$$k \leftarrow k - \alpha \frac{\partial J}{\partial k} \qquad\qquad \sigma_0 \leftarrow \sigma_0 - \alpha \frac{\partial J}{\partial \sigma_0}$$

### Implementing It

In [ ]:
def gradient_descent(x, y, lr, n_iterations, track_every=None):
    n = len(x)
    k_, sigma0_ = 0.0, 0.0
    loss_history = []
    snapshots = []

    for i in range(n_iterations):
        y_pred = sigma0_ + k_ * x
        loss = np.mean((y - y_pred) ** 2)
        loss_history.append(loss)

        if track_every and i % track_every == 0:
            snapshots.append((i, sigma0_, k_, loss))

        dk = (-2/n) * np.sum(x * (y - y_pred))
        dsigma0 = (-2/n) * np.sum(y - y_pred)
        k_ -= lr * dk
        sigma0_ -= lr * dsigma0

    return sigma0_, k_, loss_history, snapshots


In [ ]:
sigma0_fit, k_fit, losses, snapshots = gradient_descent(x, y, lr=0.001, n_iterations=100001, track_every=20000)

print(f"{'Iteration':<12}{'sigma_0':<12}{'k':<10}{'Loss':<12}")
for it, s0, k1, l in snapshots:
    print(f"{it:<12}{s0:<12.2f}{k1:<10.2f}{l:<12.2f}")

print()
print(f"Final result:  sigma_0 = {sigma0_fit:.2f}, k = {k_fit:.2f}")
print(f"np.polyfit:    sigma_0 = {sigma0_true:.2f}, k = {k_true:.2f}")


### Watching It Converge

In [ ]:
plt.figure(figsize=(6.5, 4.5))
plt.plot(losses[:20000])
plt.xlabel("Iteration")
plt.ylabel("Loss (MSE)")
plt.title("Gradient descent loss curve (lr = 0.001)")
plt.tight_layout()
plt.show()


---
## Quick Check A

Why do we update $\sigma_0$ and $k$ **simultaneously** (using their old values to compute both new values) rather than updating $k$ first and then using the *new* $k$ to update $\sigma_0$?

**(i)** It doesn't actually matter which order you use
**(ii)** Updating sequentially would mean the second update no longer follows the true gradient direction of the original point
**(iii)** Simultaneous updates are simply faster to compute

*Think about it, then check the answer below.*

**Answer: (ii)** — the gradient tells us the direction to move from a *specific* point $(\sigma_0, k)$. If we update $k$ first and then compute $\sigma_0$'s gradient using the *new* $k$, we're no longer moving in the direction the original gradient actually pointed – the math only guarantees improvement if both updates come from the same starting point.

---
## Part B — Convergence & Convexity

The loss curve above is smooth and monotonically decreasing — no bumps, no getting stuck partway. This isn't a coincidence: for linear regression, the loss function $J(\sigma_0, k)$ is always **convex** — shaped like a smooth bowl with a single global minimum, no matter what data you use.

This guarantees that gradient descent, started from *any* point, will always find its way to the bottom — the true best fit. Let's see the bowl shape directly.

In [ ]:
sigma0_range = np.linspace(0, 160, 60)
k_range = np.linspace(-10, 50, 60)
S0, K = np.meshgrid(sigma0_range, k_range)

loss_surface = np.zeros_like(S0)
for i in range(S0.shape[0]):
    for j in range(S0.shape[1]):
        y_pred = S0[i, j] + K[i, j] * x
        loss_surface[i, j] = np.mean((y - y_pred) ** 2)

plt.figure(figsize=(6.5, 5))
contour = plt.contourf(S0, K, loss_surface, levels=30, cmap="viridis")
plt.colorbar(contour, label="Loss (MSE)")
plt.scatter([sigma0_true], [k_true], color="red", marker="*", s=200, label="True minimum (np.polyfit)")
plt.xlabel("sigma_0")
plt.ylabel("k")
plt.title("The loss surface is a single smooth bowl (convex)")
plt.legend()
plt.tight_layout()
plt.show()


**Why this matters going forward:** this convexity guarantee is specific to linear regression. Later this week (Day 4, neural networks), the loss surface is *not* convex — it can have multiple local minima, flat regions, and other complications. That's part of why training a neural network is harder to reason about than fitting a line.

---
## Quick Check B

If linear regression's loss surface has a single global minimum, does the *starting point* we choose for $(\sigma_0, k)$ matter?

**(i)** Yes, a bad starting point means gradient descent might never find the true minimum
**(ii)** No, gradient descent will reach the same minimum regardless of starting point (assuming a suitable learning rate)
**(iii)** It matters only if the starting point is negative

*Think about it, then check the answer below.*

**Answer: (ii)** — thanks to convexity, any reasonable starting point eventually reaches the same global minimum. This is *not* true in general for non-convex problems, where a poor starting point can get stuck in a local minimum instead.

---
## Part C — Hyperparameters: Learning Rate, Batch Size, and Epochs

**Hyperparameters** are settings we choose *before* training – unlike $\sigma_0$ and $k$, they aren't learned from data. The learning rate is the one we already know matters a lot: recall our own notebook from Day 1 morning, where lr=0.01 caused the fit to diverge. Let's see that failure happen live, side by side with a learning rate that works.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # the diverging run below intentionally overflows

_, _, losses_diverge, _ = gradient_descent(x, y, lr=0.01, n_iterations=8)
_, _, losses_converge, _ = gradient_descent(x, y, lr=0.001, n_iterations=8)

print("lr = 0.01  (too large):")
for i, l in enumerate(losses_diverge):
    print(f"  iteration {i}: loss = {l:,.1f}")

print()
print("lr = 0.001 (works):")
for i, l in enumerate(losses_converge):
    print(f"  iteration {i}: loss = {l:.2f}")


Notice: with lr=0.01, the loss doesn't just fail to improve — it grows *explosively*, roughly 10x larger each iteration. This is what "too high a learning rate" actually looks like in practice, not just in theory.

### Batch Size and Epochs

Two more hyperparameters worth knowing, though our dataset is too small to demonstrate them meaningfully:

- **Batch size** – how many data points are used to compute each gradient step. With large datasets, using all points every step (*full-batch*, what we've done here) can be slow, so mini-batches (a random subset each step) or a single point at a time (*stochastic gradient descent*) are common alternatives.
- **Epochs** – one epoch means the model has seen every training example once. With 7 (or even 198) data points, there's no real benefit to mini-batching – full-batch gradient descent, as we've used, is simply the right choice at this scale.

---
## Quick Check C

Based on what we just saw, what's the safest way to choose a learning rate for a new problem?

**(i)** Always use the largest learning rate possible, for the fastest convergence
**(ii)** Start small and only increase it if convergence seems too slow, watching the loss curve for divergence
**(iii)** The learning rate doesn't matter as long as you run enough iterations

*Think about it, then check the answer below.*

**Answer: (ii)** — as we just saw directly, too large a learning rate causes the loss to explode rather than converge, regardless of how many iterations you run. Starting conservatively and watching the loss curve is the standard practical approach.

---
## Part D — Multiple Features & Feature Scaling

**Multiple features:** so far we've had one feature ($d^{-1/2}$). With several features (as we'll see from Day 2 onward, once we're using multiple materials descriptors at once), the same equation simply extends:

$$y' = b + w_1 x_1 + w_2 x_2 + \dots + w_n x_n$$

and gradient descent works exactly the same way — one partial derivative per weight, all updated simultaneously.

### Feature Scaling

Here's something worth connecting back to our own experience today: our Hall-Petch feature, $d^{-1/2}$, ranges from about 2 to 23.6 — **unscaled**. This is exactly *why* we needed such a small learning rate (0.001) to avoid diverging. Let's confirm this directly: if we scale the feature down first, does a much larger learning rate suddenly become safe?

In [ ]:
x_scaled = x / x.max()
print(f"Original feature range:  {x.min():.2f} to {x.max():.2f}")
print(f"Scaled feature range:    {x_scaled.min():.2f} to {x_scaled.max():.2f}")

sigma0_scaled, k_scaled, losses_scaled, _ = gradient_descent(x_scaled, y, lr=0.5, n_iterations=5000)

# Unscale k back to the original units (sigma_0 is unaffected by scaling x)
k_unscaled = k_scaled / x.max()

print()
print(f"Using SCALED features with lr=0.5 (500x larger than before!), only 5000 iterations:")
print(f"  sigma_0 = {sigma0_scaled:.2f}, k (unscaled) = {k_unscaled:.2f}")
print(f"  Compare to np.polyfit:  sigma_0 = {sigma0_true:.2f}, k = {k_true:.2f}")


Scaling the feature let us use a learning rate **500 times larger**, converging in a fraction of the iterations – and still landing on the exact same answer. This is the practical reason feature scaling is considered standard practice: it makes gradient descent faster and far less sensitive to learning-rate choice.

---
## Quick Check D

Why did scaling the feature allow such a dramatically larger learning rate?

**(i)** Scaling changes the true relationship between grain size and yield stress
**(ii)** With a smaller feature range, a given learning rate produces smaller, safer parameter updates – so a larger rate can be used without overshooting
**(iii)** Scaling has no real effect; the result was a coincidence

*Think about it, then check the answer below.*

**Answer: (ii)** — the size of each gradient descent step depends on the feature's scale. A feature ranging up to 23.6 produces much larger gradient values (and so much larger steps) than one scaled down to a maximum of 1.0. Scaling levels the playing field, which is why it's standard practice whenever features have very different natural ranges — especially relevant from Day 2 onward, when we'll have several descriptors with different units and scales at once.

## Wrap-Up

- Built gradient descent from scratch and confirmed it converges to the exact same answer as `np.polyfit`
- Saw *why* it's guaranteed to work for linear regression: a convex, single-minimum loss surface
- Reproduced our own real learning-rate divergence bug live, and used it to motivate careful hyperparameter choice
- Confirmed, numerically, that feature scaling allows a dramatically larger learning rate — directly explaining why our original Hall-Petch gradient descent needed such a small learning rate in the first place

**Next:** Day 1 closes with the capstone project briefing — choose a problem, and think about which of this week's techniques might apply to it.